<a href="https://colab.research.google.com/github/evgeniy-borisov/vairl/blob/main/notebooks/semantic-torrent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Torrent: торрент + векторный поиск по смыслу

Децентрализованная модель: файл разбивается на **чанки**, у каждого — **хэш содержимого** (как в BitTorrent) и **вектор смысла**. Пиры объявляют `(piece_hash, embedding)` в распределённый индекс; поиск — по косинусной близости запроса к объявлениям, скачивание — по хэшу.

**VAIRL** — Virtual AI Research Lab

## 1. Зависимости

In [ ]:
!pip install -q scikit-learn matplotlib numpy

import hashlib
import json
import re
import random
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)

## 2. Документ и семантическое разбиение

В классическом торренте куски фиксированного размера (256 KiB). Для **смыслового поиска** удобнее чанки по границам абзацев/предложений — каждый кусок несёт связную мысль и получает свой embedding.

In [ ]:
DOCUMENT = """
BitTorrent делит файл на куски фиксированного размера. Каждый кусок идентифицируется SHA-1 хэшем.
Клиент скачивает куски у пиров, проверяет хэш и собирает файл по меркле-дереву из .torrent метаданных.

Векторный поиск находит фрагменты текста по смыслу запроса, а не по точному совпадению байтов.
Embedding модели кодируют семантику в плотный вектор; близость измеряется косинусным расстоянием.

RAG-системы хранят чанки документов в vector DB и извлекают top-k по запросу пользователя.
Качество ответа агента зависит от гранулярности чанков и релевантности retrieval.

DHT в торрент-сети разрешает info_hash в список пиров, у которых есть куски файла.
Расширение: пиры дополнительно публикуют embedding каждого куска в семантический индекс.

Семантический торрент объединяет content-addressed хранение и ANN-поиск по смыслу.
Запрос «найди куски про векторный поиск» возвращает piece_hash; дальше — обычный обмен по протоколу.

Децентрализация: нет единого сервера индекса; каждый пир хранит подмножество кусков и объявляет их в сеть.
Поисковые узлы могут агрегировать объявления или использовать locality-sensitive hashing по embedding.
""".strip()

def semantic_chunk(text: str, max_chars: int = 180) -> List[str]:
    """Разбиение по абзацам, затем по предложениям при переполнении."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks: List[str] = []
    for para in paragraphs:
        if len(para) <= max_chars:
            chunks.append(para)
            continue
        sentences = re.split(r"(?<=[.!?])\s+", para)
        buf = ""
        for sent in sentences:
            if len(buf) + len(sent) + 1 <= max_chars:
                buf = (buf + " " + sent).strip()
            else:
                if buf:
                    chunks.append(buf)
                buf = sent
        if buf:
            chunks.append(buf)
    return chunks

chunks = semantic_chunk(DOCUMENT)
print(f"Чанков: {len(chunks)}\n")
for i, c in enumerate(chunks):
    print(f"[{i}] ({len(c)} симв.) {c[:72]}...")

## 3. Хэш куска + вектор смысла

| Поле | Назначение |
|------|------------|
| `piece_hash` | Целостность и адресация (как в BitTorrent) |
| `embedding` | Семантический поиск и кластеризация |
| `index` | Порядок сборки файла |

In [ ]:
@dataclass
class SemanticPiece:
    index: int
    text: str
    piece_hash: str
    embedding: np.ndarray

def piece_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

vectorizer = TfidfVectorizer(max_features=128, ngram_range=(1, 2))
matrix = vectorizer.fit_transform(chunks)
embeddings = matrix.toarray().astype(np.float32)

pieces: List[SemanticPiece] = []
for i, (text, emb) in enumerate(zip(chunks, embeddings)):
  pieces.append(SemanticPiece(i, text, piece_hash(text), emb))

info_hash = hashlib.sha256("".join(p.piece_hash for p in pieces).encode()).hexdigest()[:20]
print(f"info_hash (метаданные «файла»): {info_hash}\n")
for p in pieces[:3]:
    print(f"#{p.index} hash={p.piece_hash}  dim={p.embedding.shape[0]}  «{p.text[:50]}…»")

## 4. Семантический DHT (симуляция)

Классический DHT: `info_hash → peers`. Расширение: **`embedding_bucket → [(piece_hash, peer_id)]`** или глобальный ANN-индекс поверх объявлений пиров.

In [ ]:
@dataclass
class Peer:
    peer_id: str
    piece_indices: List[int]

class SemanticDHT:
    """Упрощённый индекс: все объявления пиров + косинусный поиск."""

    def __init__(self, pieces: List[SemanticPiece]):
        self.pieces = pieces
        self.announcements: List[Tuple[str, int, str]] = []  # peer_id, index, piece_hash

    def announce(self, peer: Peer) -> None:
        for idx in peer.piece_indices:
            p = self.pieces[idx]
            self.announcements.append((peer.peer_id, idx, p.piece_hash))

    def search(self, query: str, top_k: int = 3) -> List[dict]:
        q_vec = vectorizer.transform([query]).toarray().astype(np.float32)
        sims = cosine_similarity(q_vec, embeddings)[0]
        ranked = np.argsort(-sims)[:top_k]
        results = []
        for idx in ranked:
            p = self.pieces[int(idx)]
            peers = sorted({pid for pid, i, _ in self.announcements if i == p.index})
            results.append({
                "index": p.index,
                "piece_hash": p.piece_hash,
                "similarity": float(sims[idx]),
                "text": p.text,
                "peers": peers,
            })
        return results

# Симуляция роя: каждый пир хранит случайное подмножество кусков
n_peers = 5
peers = []
for i in range(n_peers):
    k = max(2, len(pieces) // 2)
    indices = sorted(random.sample(range(len(pieces)), k))
    peers.append(Peer(f"peer-{i+1}", indices))

dht = SemanticDHT(pieces)
for peer in peers:
    dht.announce(peer)

print("Пиры и их куски:")
for p in peers:
    print(f"  {p.peer_id}: indices {p.piece_indices}")

## 5. Поиск кусков по смыслу

In [ ]:
QUERIES = [
    "векторный поиск и embedding",
    "как работает DHT в торренте",
    "RAG и чанки для агента",
]

for q in QUERIES:
    print(f"\n{'='*60}\nЗапрос: «{q}»\n")
    for r in dht.search(q, top_k=2):
        print(f"  #{r['index']} sim={r['similarity']:.3f} hash={r['piece_hash']}")
        print(f"     «{r['text'][:70]}…»")
        print(f"     пиры: {', '.join(r['peers']) or 'нет в рое'}")

## 6. Визуализация: чанки в 2D и запрос

PCA проецирует TF-IDF векторы; цвет — индекс куска; звезда — запрос.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)

query = "децентрализованный семантический индекс"
q_vec = vectorizer.transform([query]).toarray()
q_2d = pca.transform(q_vec)[0]
sims = cosine_similarity(q_vec, embeddings)[0]
best = int(np.argmax(sims))

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(pieces)))
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, s=120, c=[colors[i]], edgecolors="white", linewidths=1.2, zorder=3)
    ax.annotate(f"#{i}", (x, y), textcoords="offset points", xytext=(6, 4), fontsize=9)

ax.scatter(q_2d[0], q_2d[1], s=220, marker="*", c="#fa709a", edgecolors="#333", linewidths=0.8, zorder=5, label="запрос")
ax.scatter(coords[best, 0], coords[best, 1], s=200, facecolors="none", edgecolors="#fa709a", linewidths=2.5, zorder=4, label=f"лучший #{best}")

# пиры — кольца вокруг «своих» кусков
for peer in peers:
    px = np.mean([coords[i, 0] for i in peer.piece_indices])
    py = np.mean([coords[i, 1] for i in peer.piece_indices])
    ax.scatter(px, py, s=60, marker="s", c="#667eea", alpha=0.7, zorder=2)
    ax.annotate(peer.peer_id, (px, py), textcoords="offset points", xytext=(4, -10), fontsize=7, color="#555")

ax.set_title(f"Semantic Torrent: «{query}» → piece #{best} (sim={sims[best]:.2f})")
ax.legend(loc="upper right")
ax.set_xlabel("PCA-1"); ax.set_ylabel("PCA-2")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Метаданные .semantic-torrent

Расширение к `.torrent`: для каждого piece — `hash` + `embedding` (или ссылка на модель). Поисковый клиент: ANN по индексу → `piece_hash` → `GET_BLOCK` у пира.

In [ ]:
manifest = {
    "info_hash": info_hash,
    "piece_count": len(pieces),
    "embedding_model": "tfidf-demo",
    "pieces": [
        {
            "index": p.index,
            "piece_hash": p.piece_hash,
            "text_preview": p.text[:80],
            "embedding_dim": int(p.embedding.shape[0]),
        }
        for p in pieces
    ],
}
print(json.dumps(manifest, ensure_ascii=False, indent=2)[:1200], "\n…")